## データセットについて

本実験では、日本の企業情報を管理する2つの公的データソースを使用する。

### EDINET（Electronic Disclosure for Investors' NETwork）

金融庁が運営する **有価証券報告書等の開示書類の電子開示システム**。
上場企業だけでなく、投資信託・ファンド・非上場の届出企業など **約1万件以上** のエンティティが登録されている。
各社に一意の **EDINET コード**（例: `E00001`）が付与され、企業名は日本語・英語・カナの3表記が存在する。
証券コードは上場企業にのみ付与されるため、全レコードに存在するわけではない。

- 公式サイト: https://disclosure2.edinet-fsa.go.jp/
- 主なカラム: `EDINET_CODE`, `COMPANY_NAME_JA`, `COMPANY_NAME_EN`, `COMPANY_NAME_KANA`, `SECURITIES_CODE`, `CORPORATE_NUMBER`

### JPX（Japan Exchange Group / 日本取引所グループ）

東京証券取引所（東証）を運営する **JPX が公開する上場銘柄一覧**。
プライム・スタンダード・グロース等の市場区分、業種分類（33業種・17業種）が含まれる。
証券コード（4桁）で各銘柄を一意に識別でき、約 **4,000〜4,500 銘柄** が掲載されている。

- 公式サイト: https://www.jpx.co.jp/
- 主なカラム: `SECURITIES_CODE`, `COMPANY_NAME`, `MARKET`, `INDUSTRY_33`

### 共通キー: `SECURITIES_CODE`（証券コード）

両テーブルの結合キーは **証券コード（`SECURITIES_CODE`）** である。

| 項目 | EDINET | JPX |
|------|--------|-----|
| カラム名 | `SECURITIES_CODE` | `SECURITIES_CODE` |
| 形式 | 5桁文字列（末尾0）例: `72030` | 4桁文字列 例: `7203` |
| カバレッジ | 上場企業のみ付与（NULL多数） | 全レコードに存在 |

EDINET の証券コードは **5桁**（末尾に `0` が付加）、JPX は **4桁** であるため、結合時には桁数の正規化が必要になる。
具体的には、EDINET 側の先頭4桁を切り出すか、JPX 側に `0` を付加して揃える。
この証券コードの完全一致を **名寄せの正解ラベル（Ground Truth）** として使用する。

### 名寄せの課題

同一企業でも両データソース間で社名の表記が異なることが多い:

| パターン | JPX 表記例 | EDINET 表記例 |
|----------|-----------|---------------|
| 株式会社の位置 | マルハニチロ | 株式会社マルハニチロ |
| 略称 vs 正式名 | ＳＧＨＤ | ＳＧホールディングス株式会社 |
| 全角/半角 | ＡＮＡホールディングス | ANAホールディングス株式会社 |

この表記揺れを **Cortex Search Service のセマンティック検索** で吸収し、証券コードによる完全一致を正解データとして精度を評価する。

# 企業名名寄せ実験：EDINET × JPX（実データ）

Cortex Search Service を使って、JPX上場銘柄（JPX）とEDINETコードリスト（EDINET）を
社名の表記揺れを考慮して紐付ける実験。
SECURITIES_CODE による完全一致結合をグランドトゥルースとして、名寄せ精度を定量評価する。

## 実験フロー

| STEP | 処理 | 出力テーブル |
|------|------|-----------|
| 0 | データ確認（件数・証券コード保有率・品質チェック） | — |
| 1 | 正解データ作成（SECURITIES_CODE 結合） | `SANDBOX_DB.WORK.EDINET_JPX_GROUND_TRUTH` |
| 2 | Cortex Search Service 作成（EDINET 社名をインデックス化） | `CORTEX_DB.SEARCH_SERVICES.EDINET_COMPANY_SEARCH` |
| 3 | 単件確認（任意の社名で動作確認） | — |
| 4 | 全件マッチング実行 | `SANDBOX_DB.WORK.EDINET_JPX_MATCH_RESULT` |
| 5 | 精度評価（Precision@1 / Recall@3） | — |

---
## STEP 0: データ確認

件数・NULL率・証券コード保有率を確認して、実験の前提を把握する。

In [ ]:
%%sql -r dataframe_1
-- 各マテビューのレコード件数
SELECT 'MV_EDINET_COMPANIES' AS mv_name, COUNT(*) AS row_count FROM RAW_DB.COMPANY_MATCHING.MV_EDINET_COMPANIES
UNION ALL
SELECT 'MV_JPX_COMPANIES',               COUNT(*)               FROM RAW_DB.COMPANY_MATCHING.MV_JPX_COMPANIES
UNION ALL
SELECT 'MV_NTA_COMPANIES',               COUNT(*)               FROM RAW_DB.COMPANY_MATCHING.MV_NTA_COMPANIES;

In [ ]:
%%sql -r dataframe_2
-- EDINET: NULL集計（名寄せに使うキー列の品質確認）
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN EDINET_CODE       IS NULL THEN 1 ELSE 0 END) AS null_edinet_code,
  SUM(CASE WHEN COMPANY_NAME_JA   IS NULL OR COMPANY_NAME_JA   = '' THEN 1 ELSE 0 END) AS null_or_empty_name_ja,
  SUM(CASE WHEN COMPANY_NAME_EN   IS NULL OR COMPANY_NAME_EN   = '' THEN 1 ELSE 0 END) AS null_or_empty_name_en,
  SUM(CASE WHEN COMPANY_NAME_KANA IS NULL OR COMPANY_NAME_KANA = '' THEN 1 ELSE 0 END) AS null_or_empty_kana,
  SUM(CASE WHEN SECURITIES_CODE   IS NULL OR SECURITIES_CODE   = '' THEN 1 ELSE 0 END) AS null_or_empty_securities_code,
  SUM(CASE WHEN CORPORATE_NUMBER  IS NULL OR CORPORATE_NUMBER  = '' THEN 1 ELSE 0 END) AS null_or_empty_corporate_number
FROM RAW_DB.COMPANY_MATCHING.MV_EDINET_COMPANIES;

In [ ]:
%%sql -r dataframe_3
-- JPX: NULL集計
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN SECURITIES_CODE IS NULL OR SECURITIES_CODE = '' THEN 1 ELSE 0 END) AS null_or_empty_securities_code,
  SUM(CASE WHEN COMPANY_NAME    IS NULL OR COMPANY_NAME    = '' THEN 1 ELSE 0 END) AS null_or_empty_name,
  SUM(CASE WHEN MARKET          IS NULL OR MARKET          = '' THEN 1 ELSE 0 END) AS null_or_empty_market
FROM RAW_DB.COMPANY_MATCHING.MV_JPX_COMPANIES;

In [ ]:
%%sql -r dataframe_4
-- EDINET × JPX: SECURITIES_CODEで突合可能か確認
SELECT
  COUNT(DISTINCT e.SECURITIES_CODE) AS edinet_with_securities_code,
  COUNT(DISTINCT j.SECURITIES_CODE) AS jpx_codes,
  COUNT(DISTINCT CASE WHEN j.SECURITIES_CODE IS NOT NULL THEN e.SECURITIES_CODE END) AS matched_count
FROM RAW_DB.COMPANY_MATCHING.MV_EDINET_COMPANIES e
LEFT JOIN RAW_DB.COMPANY_MATCHING.MV_JPX_COMPANIES j
  ON LPAD(e.SECURITIES_CODE::VARCHAR, 4, '0') = LPAD(j.SECURITIES_CODE::VARCHAR, 4, '0')
WHERE e.SECURITIES_CODE IS NOT NULL AND e.SECURITIES_CODE != '';

In [ ]:
%%sql -r dataframe_5
-- EDINET: サンプル確認（社名表記の感触をつかむ）
SELECT SECURITIES_CODE, COMPANY_NAME_JA
FROM RAW_DB.COMPANY_MATCHING.MV_EDINET_COMPANIES
WHERE SECURITIES_CODE IS NOT NULL AND SECURITIES_CODE != ''
LIMIT 10;

In [ ]:
%%sql -r dataframe_6
-- JPX: サンプル確認
SELECT SECURITIES_CODE, COMPANY_NAME, MARKET
FROM RAW_DB.COMPANY_MATCHING.MV_JPX_COMPANIES
LIMIT 10;

---
## STEP 1: 正解データ作成（SECURITIES_CODE による完全一致結合）

EDINET と JPX を SECURITIES_CODE で直接結合してグランドトゥルースを作成する。
この件数が精度評価の分母（評価母数）になる。

> `LPAD(..., 4, '0')` でコードの桁数揺れを吸収する。

In [ ]:
%%sql -r dataframe_7
-- 証券コード(4桁正規化)でJPX×EDINETをINNER JOINし、正解ペア数を確認
SELECT COUNT(*) AS ground_truth_count
FROM RAW_DB.COMPANY_MATCHING.MV_JPX_COMPANIES j
INNER JOIN RAW_DB.COMPANY_MATCHING.MV_EDINET_COMPANIES e
    ON LPAD(j.SECURITIES_CODE::VARCHAR, 4, '0') = LPAD(e.SECURITIES_CODE::VARCHAR, 4, '0')
WHERE j.SECURITIES_CODE IS NOT NULL
  AND j.SECURITIES_CODE != '-';

In [ ]:
%%sql -r dataframe_8
-- 社名の表記差異を目視確認（名寄せの難易度感をつかむ）
SELECT
    j.SECURITIES_CODE,
    j.COMPANY_NAME       AS JPX_NAME,
    e.COMPANY_NAME_JA    AS EDINET_NAME,
    e.EDINET_CODE,
    j.INDUSTRY_33        AS JPX_INDUSTRY
FROM RAW_DB.COMPANY_MATCHING.MV_JPX_COMPANIES j
INNER JOIN RAW_DB.COMPANY_MATCHING.MV_EDINET_COMPANIES e
    ON LPAD(j.SECURITIES_CODE::VARCHAR, 4, '0') = LPAD(e.SECURITIES_CODE::VARCHAR, 4, '0')
WHERE j.SECURITIES_CODE IS NOT NULL
  AND j.SECURITIES_CODE != '-'
LIMIT 20;

In [ ]:
%%sql -r dataframe_9
-- 名寄せ検証用：社名相違が顕著な10社（株式会社の有無だけでない相違）
SELECT
    j.SECURITIES_CODE,
    j.COMPANY_NAME       AS JPX_NAME,
    e.COMPANY_NAME_JA    AS EDINET_NAME,
    e.EDINET_CODE,
    j.INDUSTRY_33        AS JPX_INDUSTRY
FROM RAW_DB.COMPANY_MATCHING.MV_JPX_COMPANIES j
INNER JOIN RAW_DB.COMPANY_MATCHING.MV_EDINET_COMPANIES e
    ON LPAD(j.SECURITIES_CODE::VARCHAR, 4, '0') = LPAD(e.SECURITIES_CODE::VARCHAR, 4, '0')
WHERE LPAD(j.SECURITIES_CODE::VARCHAR, 4, '0') IN (
    '1333', '6417', '6676', '2292', '8203',
    '8750', '6634', '7942', '6237', '3547'
)
ORDER BY j.SECURITIES_CODE;

In [ ]:
%%sql -r dataframe_10
-- グランドトゥルーステーブル作成
CREATE OR REPLACE TABLE SANDBOX_DB.WORK.EDINET_JPX_GROUND_TRUTH AS
SELECT
    j.SECURITIES_CODE,
    j.COMPANY_NAME       AS JPX_NAME,
    e.COMPANY_NAME_JA    AS EDINET_NAME,
    e.EDINET_CODE,
    e.CORPORATE_NUMBER,
    j.INDUSTRY_33        AS JPX_INDUSTRY
FROM RAW_DB.COMPANY_MATCHING.MV_JPX_COMPANIES j
INNER JOIN RAW_DB.COMPANY_MATCHING.MV_EDINET_COMPANIES e
    ON LPAD(j.SECURITIES_CODE::VARCHAR, 4, '0') = LPAD(e.SECURITIES_CODE::VARCHAR, 4, '0')
WHERE j.SECURITIES_CODE IS NOT NULL
  AND j.SECURITIES_CODE != '-';

SELECT COUNT(*) AS ground_truth_count FROM SANDBOX_DB.WORK.EDINET_JPX_GROUND_TRUTH;

---
## STEP 2: Cortex Search Service 作成

EDINET 社名（`COMPANY_NAME_JA`）をベクトルインデックス化する。
JPX 社名をクエリとして投入するため **EDINET 側を索引化** する。

> `ACTIVE` になるまで数分かかる場合がある。次のセルで状態を確認してから STEP 3 へ進む。

In [ ]:
%%sql -r ctas_result
-- ============================================================
-- 検索用テーブルの作成 (CTAS: CREATE TABLE AS SELECT)
-- ============================================================
-- なぜ必要？
--   Cortex Search Service は内部で「Dynamic Table」という仕組みを使って
--   データを自動更新するが、Dynamic Table は外部テーブルや
--   マテリアライズドビュー(MV)をソースにできない制約がある。
--   そのため、MV のデータを通常テーブルにコピーしてから使う。
-- ============================================================
CREATE TABLE IF NOT EXISTS SANDBOX_DB.WORK.EDINET_COMPANIES_FOR_SEARCH AS
SELECT COMPANY_NAME_JA, EDINET_CODE, SECURITIES_CODE, CORPORATE_NUMBER, COMPANY_NAME_KANA
FROM RAW_DB.COMPANY_MATCHING.MV_EDINET_COMPANIES
WHERE COMPANY_NAME_JA IS NOT NULL;

In [ ]:
%%sql -r create_search_service_result
-- ============================================================
-- Cortex Search Service の作成
-- ============================================================
-- Cortex Search とは？
--   Snowflake 組み込みのセマンティック検索サービス。
--   テキストを自動でベクトル化（Embedding）し、
--   意味的に近い文字列を高速に検索できる。
--
-- 主なパラメータ:
--   ON ...           → 検索対象のテキストカラム（ここでは日本語社名）
--   ATTRIBUTES ...   → 検索結果と一緒に返したいカラム
--   TARGET_LAG       → ソーステーブル更新後、何時間以内にインデックスを更新するか
--   WAREHOUSE        → インデックス構築に使うウェアハウス
--
-- 実行には CREATE CORTEX SEARCH SERVICE 権限が必要（SYSADMIN で実行）
-- ============================================================
CREATE OR REPLACE CORTEX SEARCH SERVICE CORTEX_DB.SEARCH_SERVICES.EDINET_COMPANY_SEARCH
    ON COMPANY_NAME_JA
    ATTRIBUTES EDINET_CODE, SECURITIES_CODE, CORPORATE_NUMBER, COMPANY_NAME_KANA
    WAREHOUSE = SANDBOX_WH
    TARGET_LAG = '1 hour'
    AS (
        SELECT COMPANY_NAME_JA, EDINET_CODE, SECURITIES_CODE, CORPORATE_NUMBER, COMPANY_NAME_KANA
        FROM SANDBOX_DB.WORK.EDINET_COMPANIES_FOR_SEARCH
    );

In [ ]:
%%sql -r dataframe_11
-- Cortex Search は内部的に Dynamic Table を使うため、外部テーブル参照の MV は使えない
-- 事前に SYSADMIN で CTAS → Search Service 作成済み。ここでは状態を確認する
SELECT
    'EDINET_COMPANIES_FOR_SEARCH' AS source_table,
    (SELECT COUNT(*) FROM SANDBOX_DB.WORK.EDINET_COMPANIES_FOR_SEARCH) AS row_count;


In [ ]:
%%sql -r dataframe_12
-- ACTIVE になるまで待機（status が ACTIVE になったら STEP 3 へ）
SHOW CORTEX SEARCH SERVICES IN DATABASE CORTEX_DB;

---
## STEP 3: 単件確認

任意の JPX 社名を `query` に指定して動作確認する。

> **確認ポイント**: RANK=1 の候補が正しい EDINET エントリを指しているか、SCORE の目安値を確認する。

In [ ]:
%%sql -r dataframe_13
-- 生 JSON で確認（動作確認）
SELECT PARSE_JSON(
    SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'CORTEX_DB.SEARCH_SERVICES.EDINET_COMPANY_SEARCH',
        '{
            "query": "トヨタ自動車",
            "columns": ["COMPANY_NAME_JA", "EDINET_CODE", "SECURITIES_CODE"],
            "limit": 3
        }'
    )
):results AS candidates;

In [ ]:
%%sql -r dataframe_14
-- 見やすく行展開したバージョン（"query" の社名を書き換えて試す）
WITH raw AS (
    SELECT PARSE_JSON(
        SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
            'CORTEX_DB.SEARCH_SERVICES.EDINET_COMPANY_SEARCH',
            '{"query": "トヨタ自動車", "columns": ["COMPANY_NAME_JA","EDINET_CODE","SECURITIES_CODE"], "limit": 3}'
        )
    ):results AS results
)
SELECT
    f.value:COMPANY_NAME_JA::VARCHAR  AS EDINET_NAME,
    f.value:EDINET_CODE::VARCHAR      AS EDINET_CODE,
    f.value:SECURITIES_CODE::VARCHAR  AS SECURITIES_CODE,
    f.value:score::FLOAT              AS SCORE,
    ROW_NUMBER() OVER (ORDER BY f.value:score::FLOAT DESC) AS RANK
FROM raw, LATERAL FLATTEN(INPUT => results) f;

---
## STEP 4: 全件マッチング

JPX 全銘柄に対して EDINET を Cortex Search し、上位3候補を取得する。

> **注意**: 4,440件 × Cortex Search のため実行に数分かかる。
> まず10件のサンプルで動作確認してから全件実行すること。

In [ ]:
SAMPLE_LIMIT = 10

In [ ]:
%%sql -r dataframe_15
-- ============================================================
-- Cortex Search で名寄せ実行（サンプル {{SAMPLE_LIMIT}} 件で動作確認）
-- ============================================================
-- 処理の流れ:
--   1. JPX の銘柄名（COMPANY_NAME）を検索クエリとして使う
--   2. CORTEX_SEARCH_BATCH() で EDINET 側のベクトルインデックスをバッチ検索
--   3. 類似度スコア上位3件を候補として取得
--   4. METADATA$RANK で各銘柄ごとのスコア順ランクを参照
--
-- ポイント:
--   - SEARCH_PREVIEW() はリテラル専用（カラム参照不可）のため、
--     バッチ名寄せには CORTEX_SEARCH_BATCH テーブル関数を使う
--   - TABLE(...) をカンマ結合することで各行に対して検索結果を結合
--   - MATCH_RANK = 1 が最も類似度が高い候補
-- ============================================================
SELECT
    q.SECURITIES_CODE                   AS JPX_SECURITIES_CODE,
    q.COMPANY_NAME                      AS JPX_NAME,
    r.COMPANY_NAME_JA                   AS MATCHED_EDINET_NAME,
    r.EDINET_CODE                       AS MATCHED_EDINET_CODE,
    r.SECURITIES_CODE                   AS MATCHED_SECURITIES_CODE,
    r.METADATA$RANK                     AS MATCH_RANK
FROM (
    SELECT SECURITIES_CODE, COMPANY_NAME
    FROM RAW_DB.COMPANY_MATCHING.MV_JPX_COMPANIES
    WHERE SECURITIES_CODE IS NOT NULL AND SECURITIES_CODE != '-'
    LIMIT {{SAMPLE_LIMIT}}
) AS q,
TABLE(CORTEX_SEARCH_BATCH(
    service_name => 'CORTEX_DB.SEARCH_SERVICES.EDINET_COMPANY_SEARCH',
    query        => q.COMPANY_NAME,
    limit        => 3
)) AS r
ORDER BY JPX_SECURITIES_CODE, MATCH_RANK;

In [ ]:
%%sql -r dataframe_16
-- 全件マッチング実行（グランドトゥルース全企業対象、数分かかる）
CREATE OR REPLACE TABLE SANDBOX_DB.WORK.EDINET_JPX_MATCH_RESULT AS
SELECT
    gt.SECURITIES_CODE                  AS JPX_SECURITIES_CODE,
    gt.JPX_NAME                         AS JPX_NAME,
    gt.EDINET_NAME                      AS GT_EDINET_NAME,
    r.COMPANY_NAME_JA                   AS MATCHED_EDINET_NAME,
    r.EDINET_CODE                       AS MATCHED_EDINET_CODE,
    r.SECURITIES_CODE                   AS MATCHED_SECURITIES_CODE,
    r.METADATA$RANK                     AS MATCH_RANK,
    CASE
        WHEN LPAD(r.SECURITIES_CODE::VARCHAR, 4, '0')
           = LPAD(gt.SECURITIES_CODE::VARCHAR, 4, '0')
        THEN 'CORRECT'
        ELSE 'WRONG'
    END                                 AS JUDGEMENT
FROM SANDBOX_DB.WORK.EDINET_JPX_GROUND_TRUTH AS gt,
TABLE(CORTEX_SEARCH_BATCH(
    service_name => 'CORTEX_DB.SEARCH_SERVICES.EDINET_COMPANY_SEARCH',
    query        => gt.JPX_NAME,
    limit        => 3
)) AS r;

SELECT COUNT(DISTINCT JPX_SECURITIES_CODE) AS matched_companies
FROM SANDBOX_DB.WORK.EDINET_JPX_MATCH_RESULT;

In [ ]:
%%sql -r dataframe_17
-- 結果サンプル確認
SELECT * FROM SANDBOX_DB.WORK.EDINET_JPX_MATCH_RESULT
WHERE MATCH_RANK <= 2;

In [ ]:
%%sql -r dataframe_21
-- 表記差異が顕著な10社の名寄せ結果を確認
SELECT *
FROM SANDBOX_DB.WORK.EDINET_JPX_MATCH_RESULT
WHERE MATCH_RANK <= 2
        JPX_SECURITIES_CODE IN (
    '1333', '6417', '6676', '2292', '8203',
    '8750', '6634', '7942', '6237', '3547'
)
ORDER BY JPX_SECURITIES_CODE, MATCH_RANK;

---
## STEP 5: 精度評価

| 指標 | 計算式 | 意味 |
|------|--------|------|
| **Precision@1** | 正解件数（RANK=1 かつ一致） ÷ 評価母数 | 1位の候補だけを使った場合の正解率 |
| **Recall@3** | 上位3件以内に正解が含まれた件数 ÷ 評価母数 | 候補3件提示したときに正解が含まれる率 |

> 評価母数 = `EDINET_JPX_GROUND_TRUTH` の件数（SECURITIES_CODE で紐付けできた上場銘柄数）

In [ ]:
%%sql -r dataframe_18
-- Precision@1（ベストマッチが正解かどうか）
WITH best AS (
    SELECT * FROM SANDBOX_DB.WORK.EDINET_JPX_MATCH_RESULT
    WHERE MATCH_RANK = 1
),
eval AS (
    SELECT
        g.SECURITIES_CODE,
        g.JPX_NAME,
        g.EDINET_NAME         AS CORRECT_NAME,
        b.MATCHED_EDINET_NAME AS PREDICTED_NAME,
        b.RELEVANCE_SCORE,
        CASE
            WHEN LPAD(g.SECURITIES_CODE::VARCHAR, 4, '0') = LPAD(b.MATCHED_SECURITIES_CODE::VARCHAR, 4, '0')
            THEN 'CORRECT'
            ELSE 'INCORRECT'
        END AS RESULT
    FROM SANDBOX_DB.WORK.EDINET_JPX_GROUND_TRUTH g
    LEFT JOIN best b
        ON LPAD(g.SECURITIES_CODE::VARCHAR, 4, '0') = LPAD(b.JPX_SECURITIES_CODE::VARCHAR, 4, '0')
)
SELECT
    RESULT,
    COUNT(*)                                         AS CNT,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS PCT
FROM eval
GROUP BY RESULT
ORDER BY CNT DESC;

In [ ]:
%%sql -r dataframe_19
-- 誤マッチの内容確認（INCORRECT パターン把握）
-- 高スコアなのに誤マッチしているケースが特に注目
WITH best AS (
    SELECT * FROM SANDBOX_DB.WORK.EDINET_JPX_MATCH_RESULT WHERE MATCH_RANK = 1
)
SELECT
    g.SECURITIES_CODE,
    g.JPX_NAME,
    g.EDINET_NAME          AS CORRECT,
    b.MATCHED_EDINET_NAME  AS PREDICTED,
    b.RELEVANCE_SCORE
FROM SANDBOX_DB.WORK.EDINET_JPX_GROUND_TRUTH g
JOIN best b
    ON LPAD(g.SECURITIES_CODE::VARCHAR, 4, '0') = LPAD(b.JPX_SECURITIES_CODE::VARCHAR, 4, '0')
WHERE LPAD(g.SECURITIES_CODE::VARCHAR, 4, '0') != LPAD(b.MATCHED_SECURITIES_CODE::VARCHAR, 4, '0')
ORDER BY b.RELEVANCE_SCORE DESC
LIMIT 20;

In [ ]:
%%sql -r dataframe_20
-- Recall@3（正解が上位3候補に含まれる率）
WITH top3 AS (
    SELECT * FROM SANDBOX_DB.WORK.EDINET_JPX_MATCH_RESULT
    WHERE MATCH_RANK <= 3
),
hit AS (
    SELECT
        g.SECURITIES_CODE,
        MAX(
            CASE WHEN LPAD(g.SECURITIES_CODE::VARCHAR, 4, '0') = LPAD(t.MATCHED_SECURITIES_CODE::VARCHAR, 4, '0')
                 THEN 1 ELSE 0 END
        ) AS IN_TOP3
    FROM SANDBOX_DB.WORK.EDINET_JPX_GROUND_TRUTH g
    JOIN top3 t
        ON LPAD(g.SECURITIES_CODE::VARCHAR, 4, '0') = LPAD(t.JPX_SECURITIES_CODE::VARCHAR, 4, '0')
    GROUP BY g.SECURITIES_CODE
)
SELECT
    SUM(IN_TOP3)                              AS HIT,
    COUNT(*)                                   AS TOTAL,
    ROUND(SUM(IN_TOP3) * 100.0 / COUNT(*), 1) AS RECALL_AT_3_PCT
FROM hit;